# B3 — Monte Carlo NAV Projection

This bonus analysis projects NAV growth over **5 years** using a geometric Brownian motion model calibrated from the project's cleaned NAV history. The projection uses business-day observations, 252 trading days per year, 10,000 simulations per selected scheme, and 5th/95th percentile uncertainty bands.

> **Important:** Monte Carlo projections are scenario estimates, not forecasts or investment advice. Results are highly sensitive to historical drift and volatility assumptions.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT_DIR = Path.cwd().resolve()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent
SCRIPTS_DIR = ROOT_DIR / "scripts"
sys.path.insert(0, str(SCRIPTS_DIR))
from monte_carlo import run


In [ ]:
projection, summary = run(top_n=5, years=5, simulations=10_000)
summary

## Interpretation

- The **median NAV** is the central simulated path.
- The **5th–95th percentile band** represents a wide uncertainty range across simulations.
- The model uses historical daily log-return drift and volatility, so high historical volatility produces wider bands.
- The probability of positive growth is the share of simulated terminal NAVs above the starting NAV.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 7))
for scheme_name, frame in projection.groupby("scheme_name"):
    x = frame["trading_day"] / 252
    line = ax.plot(x, frame["nav_median"], linewidth=2, label=scheme_name)[0]
    ax.fill_between(x, frame["nav_p05"], frame["nav_p95"], alpha=0.08, color=line.get_color())
ax.set_title("5-Year Monte Carlo NAV Projection", fontsize=20, fontweight="bold", loc="left")
ax.set_xlabel("Years")
ax.set_ylabel("Projected NAV")
ax.grid(alpha=0.18)
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()